# Train baseline and SFT-filtered chatbot models

This notebook currently trains two GPT-2 conditions: a baseline exposed to reports with fictional PII, and supervised fine-tuning (SFT) after removing PII-containing report examples. Both retain the same general 311 Q&A. Resident prompts are context only; loss is on assistant replies.

DPO preference data is generated, but DPO and GRPO training code is not yet implemented. See the README for the full methods and evaluation plan. The old training checkpoints and losses used a different objective; these runs save to new folders.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data" / "synthetic" / "sft_filtered_training_conversations.jsonl").exists(), (
    "Run notebook 01 or python src/generate_data.py first."
)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Uncomment in Colab or a fresh environment if dependencies are missing.
# %pip install -r requirements.txt

## Configure and run

`TRAIN_EXAMPLES` limits the number of report examples per condition; general Q&A examples are always retained. The PII-filtered condition may have fewer available reports because PII examples are removed. Set it to `None` for each condition's complete training set. Loss metrics append to `results/training_metrics.csv`, one row per model per epoch.

In [ ]:
import subprocess
from datetime import datetime

TRAIN_EXAMPLES = 250  # Report examples per condition; set None for all available.
EPOCHS = 1
RUN_LABEL = f"n{TRAIN_EXAMPLES or 'all'}-e{EPOCHS}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
command = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "train_models.py"),
    "--data-dir", str(PROJECT_ROOT / "data" / "synthetic"),
    "--model-name", "gpt2",
    "--output-dir", str(PROJECT_ROOT / "models" / "four-approach-study" / RUN_LABEL),
    "--metrics-csv", str(PROJECT_ROOT / "results" / "training_metrics.csv"),
    "--seed", "42",
    "--epochs", str(EPOCHS),
    "--batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--max-length", "128",
]
if TRAIN_EXAMPLES is not None:
    command.extend(["--max-train-examples", str(TRAIN_EXAMPLES)])
print(f"Training baseline and SFT-filtered models: report limit={TRAIN_EXAMPLES}, epochs={EPOCHS}")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Suggested comparisons

For the two implemented conditions, first use 250 report examples for one epoch as a small diagnostic; then use all available reports for one epoch; then all available reports for two epochs. Compare validation loss and later validation privacy/utility results. Keep other settings fixed and don't tune against the test set. These are exploratory runs, not independent random-seed replications.

DPO and GRPO should only be included in the comparison after their training loops and reward/preference evaluations are implemented and checked. Use the same held-out test suite for all four final methods.